In [1]:
import sys as __harness_sys
from pathlib import Path as __harness_Path
__harness_sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\tutorials')
from _offline import enforce_offline as __enforce_offline, blocked_attempts as __blocked_attempts
__enforce_offline()
import agentic_systems as __harness_ag
assert __harness_ag.__version__ == '2.1.3'
assert 'site-packages' in __harness_Path(__harness_ag.__file__).parts


# RunResult, límites y preparación

Resultado serializable y linaje en todos los límites de ejecución.

API **2.1.3**. Cada ejemplo muestra una entrada y su resultado. Python Runtime realiza trabajo determinista.

In [2]:
import agentic_systems as ag


In [3]:
@ag.tool
def increment(value: int) -> dict:
    """Suma uno de forma determinista."""
    return {"value": value + 1}

@ag.tool
def double(value: int) -> dict:
    """Duplica el valor."""
    return {"value": value * 2}

arithmetic = ag.skill(name="arithmetic", tools=[increment, double])
increase = ag.Action("increase", "increment")
multiply = ag.Action("multiply", "double")

In [4]:
import json
agent = ag.Agent("calculator", ag.Sequence((increase, multiply)), skills=(arithmetic,))
compiled = agent.compile()
result = compiled.run({"value": 3}, limits=ag.Limits(max_actions=2, timeout_s=10))
assert result.ok
encoded = json.loads(json.dumps(result.to_dict(), allow_nan=False))
assert ag.RunResult(**encoded).data == {"value": 8}
for node in result.walk():
    for child in node.children:
        assert child.parent_execution_id == node.execution_id
limited = agent.run({"value": 3}, limits=ag.Limits(max_actions=1))
assert not limited.ok
print({"output": result.data, "lineage_nodes": len(list(result.walk())), "limit_detected": not limited.ok})

{'output': {'value': 8}, 'lineage_nodes': 6, 'limit_detected': True}


In [5]:
assert (await agent.arun({'value':3})).data == result.data
assert (await compiled.arun({'value':3})).data == result.data
assert result.check_invariants().ok
assert result.trace('full') == result.to_dict()
memory = result.lineage(name='calculator',question='What happened?',goal='Explain',tags=['offline'],metadata={'tutorial':7},max_tool_rows=2)
print(memory.explain())
print(memory.to_prompt_context(max_chars=500))
print(result.trace())

{'answer': '{"value": 8}', 'ok': True, 'steps': [{'step_id': 'run-7e693ac8e34942489cc8430d56f73f3c', 'kind': 'execution', 'title': 'calculator', 'summary': '', 'source': 'python-runtime', 'why': 'Recorded execution evidence', 'evidence': {'ok': True, 'data': {'value': 8}, 'errors': []}}, {'step_id': 'run-21650f020f7740bb9e525adcd42ea0bb', 'kind': 'execution', 'title': 'sequence', 'summary': '', 'source': 'python-runtime', 'why': 'Recorded execution evidence', 'evidence': {'ok': True, 'data': {'value': 8}, 'errors': []}}, {'step_id': 'run-4f8e7e9c14da4883908b5ded537639cb', 'kind': 'execution', 'title': 'increase', 'summary': '', 'source': 'python-runtime', 'why': 'Recorded execution evidence', 'evidence': {'ok': True, 'data': {'value': 4}, 'errors': []}}, {'step_id': 'run-f6326d4ea82f4e8a923589cc704a24b3', 'kind': 'execution', 'title': 'tool', 'summary': '', 'source': 'python-runtime', 'why': 'Recorded execution evidence', 'evidence': {'ok': True, 'data': {'value': 4}, 'errors': []}}, {

to_dict es el esquema de intercambio; RunResult(**payload) restaura ese contrato. walk recorre la jerarquía, trace resume ejecución y lineage proyecta evidencia para explicación. Los contadores de Tools e inferencia son distintos. raw_responses no contiene objetos SDK ni credenciales. Los timeouts son cooperativos, no cancelación fuerte.

`compile()` no ejecuta Tools ni condiciones. `run()` lo llama internamente. Ambos caminos devuelven el mismo contrato; compile es preparación opcional, no otro constructor. Los límites son cooperativos y comparten reservas entre componentes; no matan un callback Python bloqueado. Agent/System también ofrecen `arun` con el mismo resultado.

In [6]:
assert __blocked_attempts() == 0